# Laboratorio 09 — Neo4j II: caminos, agregaciones y carga masiva

**Procesamiento Masivo de Datos (ELE051-B / EIN102B)** · USM 2026-2 · Paralelo 701
**Clase 12** · Viernes 30 de octubre de 2026 · **Laboratorio para trabajar en casa**

---

En la Clase 12 vimos la teoría y después estos ejercicios en vivo. Ahora les toca a ustedes, a su ritmo: cuenten con 1,5–2 horas. No se entrega ni se evalúa. Parte del mismo grafo del Lab 08, así que conviene haberlo hecho antes.

En el Lab 08 preguntábamos **a qué paradas llego**. Hoy la pregunta sube un nivel: **cuál es la mejor ruta** de A a B, qué paradas **no** alcanzo, cómo decide Neo4j por dónde empezar a buscar y cómo se carga un archivo **sin duplicar nada** si se carga dos veces.

Pregunta guía: *¿el camino con menos paradas es siempre el más rápido?*

**Índice:** Parte 0 (levantar Neo4j) · Parte 1 (partir del grafo de la C11) · Ejercicios 1–5 · Parte 2 (carga desde CSV) · Ejercicio 6 · Checkpoint · Síntesis.

## Cómo trabajar este notebook

- Solos o en equipos de 2–3 personas.
- **Las consultas son Cypher**, el lenguaje de Neo4j. Se escriben como texto dentro de `q("…")` y Python las envía a la base por el protocolo Bolt (puerto 7687) con el driver oficial `neo4j`. El resultado vuelve como tabla (`DataFrame`).
- **El Neo4j Browser** (<http://localhost:7474>, usuario `neo4j`, clave `pmd2026neo`) es la otra puerta a la misma base: ahí se pega el mismo Cypher y se ve **dibujado** (los caminos y los planes). Conviene tenerlo abierto en otra pestaña.
- Cada ejercicio tiene dos momentos: **«Lo de la diapositiva»** (completo: ejecútenlo y miren) y **«Ahora ustedes»** (completar el `___`, guiándose por la pista).
- Los CSV de la Parte 2 ya vienen en `import/`; no hay nada que descargar.
- Referencia: [Cypher Manual](https://neo4j.com/docs/cypher-manual/current/) · [driver de Python](https://neo4j.com/docs/python-manual/current/).
- ⚠️ La imagen es la misma del Lab 08 (`docker images | grep neo4j`). Si no la tienen: `docker pull neo4j:5-community` **antes de abrir el notebook** (~350 MB de descarga, ~1 GB en disco).
- ⚠️ Si el contenedor del Lab 08 sigue arriba, bájenlo primero (`docker compose down -v` en `08/`): es el mismo contenedor.

## Dependencias

Instala lo de `requirements.txt` (el driver `neo4j`, `pandas`, `matplotlib` y `networkx`) en **el Python de este kernel**, y confirma que se importa. Si ya estaba todo, solo dice `Requirement already satisfied` o nada. Es `%pip`, no `!pip`: `%pip` instala en el ambiente del kernel, aunque no sea el de la terminal.

In [ ]:
%pip install -q -r requirements.txt
import matplotlib, neo4j, networkx, pandas
print("✓ neo4j", neo4j.__version__, "· pandas", pandas.__version__, "· matplotlib", matplotlib.__version__, "· networkx", networkx.__version__)

## Configuración

Ejecútenla una vez. Las dos primeras funciones son las del Lab 08; las otras dos son nuevas:

- `q()` envía una consulta y devuelve un `DataFrame`; `cargar()` ejecuta el script `01_cargar_grafo.cypher` sentencia por sentencia.
- `operadores()` ejecuta una consulta con `PROFILE` y devuelve **el plan** como tabla: qué operadores usó Neo4j, cuántas filas pasaron por cada uno y cuántos **db hits** (accesos a la base) costó. Se lee de la raíz (arriba) a las hojas (abajo, donde se empieza a buscar).
- `nombres()` imprime una ruta (la lista de nombres de un camino) como `A → B → C`.

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "pmd2026neo")
driver = None   # se crea en la Parte 0


def q(cypher, **params):
    """Ejecuta una consulta Cypher y devuelve el resultado como DataFrame."""
    with driver.session() as s:
        return s.run(cypher, params).to_df()


def cargar(ruta="01_cargar_grafo.cypher"):
    """Ejecuta el script sentencia por sentencia (cada una en su propia transacción)."""
    texto = Path(ruta).read_text(encoding="utf-8")
    sentencias = []
    for trozo in texto.split(";"):
        lineas = [l for l in trozo.splitlines() if l.strip() and not l.strip().startswith("//")]
        if lineas:
            sentencias.append("\n".join(lineas))
    with driver.session() as s:
        for st in sentencias:
            s.run(st).consume()
    return len(sentencias)


def operadores(cypher, **params):
    """Ejecuta la consulta con PROFILE y devuelve el plan como tabla, de la raíz a las hojas."""
    with driver.session() as s:
        plan = s.run("PROFILE " + cypher, params).consume().profile
    filas = []

    def recorrer(nodo, nivel=0):
        filas.append({
            "operador": "  " * nivel + nodo["operatorType"].split("@")[0],
            "filas": nodo.get("rows"),
            "db_hits": nodo.get("dbHits"),
        })
        for hijo in nodo.get("children", []):
            recorrer(hijo, nivel + 1)

    recorrer(plan)
    return pd.DataFrame(filas)


def nombres(df, col="ruta"):
    """Imprime una ruta (lista de nombres) como 'A → B → C'."""
    for _, fila in df.iterrows():
        print(" → ".join(fila[col]), "·", {k: v for k, v in fila.items() if k != col})

Y la función de dibujo del Lab 08, ahora con un parámetro: `dibujar()` dibuja la red completa; `dibujar(ruta)`, con `ruta` una lista de nombres de paradas, pinta **en ámbar** las paradas y los tramos de esa ruta y atenúa el resto. Las flechas tienen el color de su línea y llevan los minutos del tramo.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx


def dibujar(ruta=None):
    """Dibuja los tramos SIGUE_A, casi como un mapa. Con `ruta` (lista de nombres), la resalta en ámbar."""
    tramos = q("MATCH (a:Parada)-[r:SIGUE_A]->(b:Parada) RETURN a.nombre AS desde, b.nombre AS hasta, r.linea AS linea, r.minutos AS minutos")
    paradas = q("MATCH (p:Parada) RETURN p.nombre AS nombre, p.lon AS lon, p.lat AS lat")
    G = nx.from_pandas_edgelist(tramos, "desde", "hasta", edge_attr=True, create_using=nx.DiGraph)
    colores = dict(zip(sorted(tramos["linea"].unique()), plt.cm.tab10.colors))
    geo = {p.nombre: (0.8 * p.lon, p.lat) for p in paradas.itertuples()}   # 0,8 ≈ cos(36,8°): aquí un grado de lon mide menos que uno de lat
    pos = nx.spring_layout(G, pos=geo, iterations=10, seed=0)              # parte del mapa y separa lo amontonado

    ambar, gris = "#E0A030", "#D0D0D0"
    en_ruta = {frozenset(par) for par in zip(ruta, ruta[1:])} if ruta else set()
    nodo_color = [ambar if ruta and n in ruta else "white" for n in G]
    borde = ["black" if not ruta or n in ruta else gris for n in G]
    arista = [ambar if frozenset((a, b)) in en_ruta else (gris if ruta else colores[d["linea"]]) for a, b, d in G.edges(data=True)]
    ancho = [4 if frozenset((a, b)) in en_ruta else 2 for a, b in G.edges()]

    plt.figure(figsize=(11, 8))
    nx.draw_networkx(G, pos, node_color=nodo_color, edgecolors=borde, node_size=600, font_size=9,
                     edge_color=arista, width=ancho, arrowsize=18)
    nx.draw_networkx_edge_labels(G, pos, edge_labels={(a, b): f"{d['minutos']}′" for a, b, d in G.edges(data=True)}, font_size=8)
    if ruta:
        plt.title(f"{ruta[0]} → {ruta[-1]}: {len(ruta) - 1} saltos (en ámbar)")
    else:
        plt.legend(handles=[plt.Line2D([], [], color=c, lw=2, label=l) for l, c in colores.items()], title="Línea")
        plt.title(f"Tramos SIGUE_A · {G.number_of_nodes()} paradas")
    plt.axis("off")
    plt.show()

---
# Parte 0 — Levantar Neo4j

`up -d --wait` levanta el contenedor (`docker-compose.yml` de esta carpeta) y espera a que Neo4j acepte consultas. Neo4j corre sobre Java: la primera vez tarda 30–60 s.

Expone dos puertos:

| Puerto | Qué |
|---|---|
| `7474` | Neo4j Browser (HTTP): <http://localhost:7474> |
| `7687` | Bolt: por donde hablan el driver de Python y `cypher-shell` |

Además, monta la carpeta `import/` de aquí en `/var/lib/neo4j/import` dentro del contenedor: ahí busca `LOAD CSV` en la Parte 2.

In [ ]:
subprocess.run(["docker", "compose", "up", "-d", "--wait"], check=True)
driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("✓ Neo4j responde ·", q("CALL dbms.components() YIELD name, versions RETURN name, versions[0] AS version").to_dict("records"))

**Esperado:** `✓ Neo4j responde · [{'name': 'Neo4j Kernel', 'version': '5.26.…'}]`. Después abran <http://localhost:7474> en el navegador, conéctense con `bolt://localhost:7687`, usuario `neo4j` y clave `pmd2026neo`, y prueben `RETURN 1`.

**Si algo falla:**
- **`container name "/pmd-neo4j" is already in use`:** quedó vivo el contenedor del Lab 08 → `docker compose -f ../08/docker-compose.yml down -v` y repetir.
- **Puerto ocupado:** `docker ps` y bajen el contenedor que sobra. No cambien el compose a ciegas.
- **`up --wait` espera ~2–3 min y termina con `container pmd-neo4j is unhealthy`** (en Python, `CalledProcessError`): casi siempre es un volumen viejo con otra clave. La clave de `NEO4J_AUTH` solo se aplica al crear el volumen. Solución: `docker compose down -v` y repetir.

---
# Parte 1 — Partir del grafo de la C11

Es la misma red **sintética** del Lab 08, cargada con el mismo `01_cargar_grafo.cypher`:

- **Nodos:** `(:Parada {id, nombre, lat, lon})`, `(:Comuna {nombre})`, `(:Linea {id, nombre})`.
- **Relaciones:** `(:Parada)-[:EN]->(:Comuna)`, `(:Linea)-[:PASA_POR {orden}]->(:Parada)` y `(:Parada)-[:SIGUE_A {linea, minutos}]->(:Parada)`.
- `SIGUE_A` apunta **en el sentido de la línea** y lleva los **minutos** del tramo: hoy esos minutos son el peso del camino.

| Línea | Nombre | Paradas | Minutos por tramo |
|---|---|---|---|
| L1 | Concepción–Talcahuano | Plaza Perú → Mall del Trébol → Parque Pedro del Río → Terminal Talcahuano | 12, 8, 10 |
| L2 | Concepción–Penco | Universidad de Concepción → Plaza Perú → Barrio Norte → Plaza Penco | 6, 7, 18 |
| L3 | Collao–Lomas Coloradas | Terminal Collao → Universidad de Concepción → Plaza Chiguayante → Lomas Coloradas | 9, 15, 14 |
| L4 | Concepción–Coronel | Plaza Perú → Puente Llacolén → Lomas Coloradas → Plaza Coronel | 9, 11, 20 |

⚠️ **Esta celda borra y recarga siempre.** Los Ejercicios 1–4 suponen el grafo **sin** la ampliación de la Parte 2, y el Ejercicio 5 supone que **no** existe el índice `parada_nombre`. Si vuelven a ejecutar el notebook con el contenedor vivo, las dos cosas ya estarían hechas y las respuestas cambiarían. Por eso se borra todo, se borra el índice si existe y se carga de nuevo (las restricciones del script usan `IF NOT EXISTS`: sobreviven sin problema).

### Lo de la diapositiva

In [ ]:
q("MATCH (n) DETACH DELETE n")
q("DROP INDEX parada_nombre IF EXISTS")
print(f"{cargar()} sentencias ejecutadas")
q("MATCH (n) RETURN labels(n)[0] AS tipo, count(*) AS n ORDER BY tipo")

**Esperado:** `6 sentencias ejecutadas` y Comuna 7 · Linea 4 · Parada 12.

La red completa, para tenerla a la vista:

In [ ]:
dibujar()

---
# Ejercicio 1 — El camino con menos paradas

`shortestPath` devuelve **un** camino con el **menor número de relaciones** entre dos nodos. Se acota siempre: `*..10` (con 12 paradas, ningún camino útil necesita más). `p = …` guarda el camino en una variable; `nodes(p)` y `length(p)` lo desarman.

### Lo de la diapositiva

De **Terminal Talcahuano** (P07) a **Plaza Penco** (P10): ¿cuántos saltos?

In [ ]:
df = q("""
MATCH (a:Parada {id: 'P07'}), (b:Parada {id: 'P10'})
MATCH p = shortestPath((a)-[:SIGUE_A*..10]-(b))
RETURN [n IN nodes(p) | n.nombre] AS ruta, length(p) AS saltos
""")
nombres(df)
dibujar(df["ruta"][0])

**Esperado:** Terminal Talcahuano → Parque Pedro del Río → Mall del Trébol → Plaza Perú → Barrio Norte → Plaza Penco, **5 saltos**.

En el Browser, peguen la misma consulta cambiando el `RETURN` por `RETURN p`: se ve el camino dibujado.

### Ahora ustedes

In [ ]:
# el camino con menos paradas de Terminal Collao (P03) a Plaza Coronel (P11)
df = q("""
MATCH (a:Parada {id: 'P03'}), (b:Parada {id: 'P11'})
MATCH p = ___((a)-[:SIGUE_A*..10]-(b))
RETURN [n IN nodes(p) | n.nombre] AS ruta, length(p) AS saltos
""")   # Pista: la misma función de la diapositiva
nombres(df)

**Esperado:** 4 saltos: Terminal Collao → Universidad de Concepción → Plaza Chiguayante → Lomas Coloradas → Plaza Coronel.

**✍️ Respuesta del equipo — 1:** ¿Por qué el patrón va sin flecha (`-`) y no con `->`? `___`

---
# Ejercicio 2 — ¿Menos saltos es menos minutos?

`shortestPath` cuenta **saltos**, no minutos. Para minimizar minutos, en un grafo chico se puede listar **todos** los caminos acotados y sumar el peso de cada uno con `reduce`: parte de `t = 0` y, por cada relación `r` del camino, acumula.

### Lo de la diapositiva

De **Universidad de Concepción** (P02) a **Lomas Coloradas** (P12). (a) Por saltos:

In [ ]:
df = q("""
MATCH (a:Parada {id: 'P02'}), (b:Parada {id: 'P12'})
MATCH p = shortestPath((a)-[:SIGUE_A*..6]-(b))
RETURN [n IN nodes(p) | n.nombre] AS ruta, length(p) AS saltos
""")
nombres(df)

(b) Por minutos: todos los caminos de hasta 6 tramos, con sus minutos sumados.

In [ ]:
df = q("""
MATCH p = (a:Parada {id: 'P02'})-[:SIGUE_A*..6]-(b:Parada {id: 'P12'})
WITH p, reduce(t = 0, r IN relationships(p) | t + r.minutos) AS minutos
RETURN [n IN nodes(p) | n.nombre] AS ruta, length(p) AS saltos, minutos
ORDER BY minutos
""")
nombres(df)
for r in df["ruta"]:
    dibujar(r)

**Esperado:** (a) Universidad de Concepción → Plaza Chiguayante → Lomas Coloradas, **2 saltos**. (b) 2 filas: primero Universidad de Concepción → Plaza Perú → Puente Llacolén → Lomas Coloradas, **3 saltos y 26 min**; después la de 2 saltos, con **29 min**. El de menos saltos **no** es el más rápido.

### Ahora ustedes

In [ ]:
# Terminal Collao → Plaza Coronel, ordenado por minutos
df = q("""
MATCH p = (a:Parada {id: 'P03'})-[:SIGUE_A*..6]-(b:Parada {id: 'P11'})
WITH p, reduce(t = 0, r IN relationships(p) | ___) AS minutos
RETURN [n IN nodes(p) | n.nombre] AS ruta, length(p) AS saltos, minutos
ORDER BY minutos
""")   # Pista: en cada paso, al acumulado t se le suman los minutos del tramo r
nombres(df)

**Esperado:** 2 filas: Terminal Collao → Universidad de Concepción → Plaza Perú → Puente Llacolén → Lomas Coloradas → Plaza Coronel, **5 saltos y 55 min**; después la de 4 saltos (por Plaza Chiguayante), con **58 min**.

**✍️ Respuesta del equipo — 2:** El Ej. 1 y este dan rutas distintas para el mismo par de paradas. ¿Cuál le recomendarían a alguien que va apurado y cuál a alguien que odia los transbordos? ¿Qué le falta al modelo para responder bien la segunda? `___`

---
# Ejercicio 3 — Agregaciones

`count`, `sum`, `avg`, `collect`: el «GROUP BY» es **implícito**; lo que no se agrega, agrupa. `collect` arma una lista por grupo y `WITH` pasa el resultado al siguiente tramo de la consulta.

### Lo de la diapositiva

(a) El **recorrido** de cada línea, en orden:

In [ ]:
q("""
MATCH (l:Linea)-[r:PASA_POR]->(p:Parada)
WITH l, p, r ORDER BY l.id, r.orden
RETURN l.id AS linea, collect(p.nombre) AS recorrido
ORDER BY linea
""")

(b) Por línea, ¿cuántos tramos y cuántos minutos suma?

In [ ]:
q("""
MATCH ()-[r:SIGUE_A]->()
RETURN r.linea AS linea, count(*) AS tramos, sum(r.minutos) AS minutos_totales
ORDER BY linea
""")

**Esperado:** (a) L1 [Plaza Perú, Mall del Trébol, Parque Pedro del Río, Terminal Talcahuano] · L2 [Universidad de Concepción, Plaza Perú, Barrio Norte, Plaza Penco] · L3 [Terminal Collao, Universidad de Concepción, Plaza Chiguayante, Lomas Coloradas] · L4 [Plaza Perú, Puente Llacolén, Lomas Coloradas, Plaza Coronel]. (b) L1 3 · 30 · L2 3 · 31 · L3 3 · 38 · L4 3 · 40: la más larga es la L4.

El `ORDER BY` del `WITH` es lo que deja cada lista en el orden de la línea: `collect` junta las filas en el orden en que le llegan.

### Ahora ustedes

In [ ]:
# minutos promedio por tramo, por línea
q("""
MATCH ()-[r:SIGUE_A]->()
RETURN r.linea AS linea, round(___(r.minutos), 2) AS min_promedio
ORDER BY linea
""")   # Pista: la agregación del promedio

**Esperado:** L1 10.0 · L2 10.33 · L3 12.67 · L4 13.33.

**✍️ Respuesta del equipo — 3:** En (a), ¿qué pasa si se quita `ORDER BY l.id, r.orden` del `WITH`? `___`

---
# Ejercicio 4 — Lo que NO se alcanza

`NOT EXISTS { MATCH … }` pregunta por la **ausencia** de un patrón. Es la pregunta típica de cobertura y accesibilidad: ¿qué queda lejos? En PostGIS, «lejos» eran metros (`ST_DWithin`); aquí son tramos.

### Lo de la diapositiva

Desde **Plaza Coronel** (P11), ¿qué paradas **no** alcanzo con hasta 3 tramos? (`q <> o` excluye la propia parada de partida.)

In [ ]:
q("""
MATCH (o:Parada {id: 'P11'})
MATCH (q:Parada)
WHERE q <> o
  AND NOT EXISTS { MATCH (o)-[:SIGUE_A*..3]-(q) }
RETURN q.nombre AS parada ORDER BY parada
""")

**Esperado:** 6 filas: Barrio Norte, Mall del Trébol, Parque Pedro del Río, Plaza Penco, Terminal Collao y Terminal Talcahuano.

### Ahora ustedes

In [ ]:
# desde Plaza Perú, ¿qué paradas NO alcanzo con a lo más 2 tramos?
q("""
MATCH (o:Parada {id: 'P01'})
MATCH (q:Parada)
WHERE q <> o
  AND NOT ___ { MATCH (o)-[:SIGUE_A*..2]-(q) }
RETURN q.nombre AS parada ORDER BY parada
""")   # Pista: la palabra que pregunta si un patrón existe

**Esperado:** Plaza Coronel y Terminal Talcahuano.

**✍️ Respuesta del equipo — 4:** Plaza Perú es el hub de la red y aun así deja 2 paradas fuera de 2 tramos. ¿Qué tienen en común esas dos? `___`

*Buen punto para una pausa: si lo hacen en dos sesiones, pueden dejar el contenedor arriba (`docker compose stop` / `start` conserva el grafo).*

---
# Ejercicio 5 — Leer un plan con PROFILE e índice

`PROFILE` ejecuta la consulta y cuenta, por operador, las filas y los **db hits**. Recorrer de un nodo a su vecino es barato (punteros directos); lo caro es **encontrar el nodo de partida**. Con 12 paradas no verán diferencias de tiempo: miren el **operador** y los **db hits**.

En el Browser, peguen las mismas consultas con `PROFILE` delante: se ve el plan dibujado.

### Lo de la diapositiva

1. Buscar Plaza Perú **por nombre**, sin índice sobre `Parada.nombre`:

In [ ]:
operadores("MATCH (p:Parada {nombre: 'Plaza Perú'}) RETURN p")

**Esperado:** `ProduceResults` ← `Filter` ← `NodeByLabelScan`: mira **todas** las paradas y filtra.

2. Crear el índice, esperar a que esté listo y mirar la lista de índices:

In [ ]:
q("CREATE INDEX parada_nombre IF NOT EXISTS FOR (p:Parada) ON (p.nombre)")
q("CALL db.awaitIndexes()")
q("SHOW INDEXES YIELD name, state, labelsOrTypes, properties")

**Esperado:** `parada_nombre` en estado `ONLINE`, junto con los 3 índices de las restricciones (`comuna_nombre`, `linea_id`, `parada_id`) y los dos índices de búsqueda por etiqueta y por tipo de relación que Neo4j trae por defecto.

3. La misma búsqueda, otra vez:

In [ ]:
operadores("MATCH (p:Parada {nombre: 'Plaza Perú'}) RETURN p")

**Esperado:** `ProduceResults` ← `NodeIndexSeek`: va directo a la parada, con menos db hits.

### Ahora ustedes

In [ ]:
# la comuna Penco, buscada por nombre, SIN crear ningún índice nuevo
operadores("MATCH (c:Comuna {nombre: ___}) RETURN c")   # Pista: el nombre de la comuna, entre comillas simples

**Esperado:** `NodeUniqueIndexSeek`.

**✍️ Respuesta del equipo — 5:** Nadie creó un índice sobre `Comuna.nombre`. ¿De dónde salió? `___`

---
# Parte 2 — Cargar una ampliación desde CSV

Hasta ahora el grafo salió de un script con `CREATE`. Con 20.000 paradas, los datos vienen en archivos. `LOAD CSV WITH HEADERS FROM 'file:///archivo.csv' AS row` lee un archivo y entrega una fila (`row`) por línea, como un mapa `{columna: valor}`.

- `file:///…` se busca en `/var/lib/neo4j/import`, dentro del contenedor. El compose monta ahí la carpeta `import/` de este laboratorio.
- **Todo llega como texto:** se convierte con `toInteger` y `toFloat`.
- `MERGE` crea **solo si no existe**: cargar dos veces deja el grafo igual (**idempotencia**). `ON CREATE SET` pone propiedades solo al crear.

La ampliación: una parada nueva, **P13 Hospital Regional**, y una línea exprés **L5** entre Terminal Talcahuano y Plaza Penco que pasa por ella.

| Archivo | Qué trae |
|---|---|
| `paradas_ext.csv` | P13 (nueva) y P07, P10 (ya existen) |
| `lineas_ext.csv` | la línea L5 |
| `pasa_por_ext.csv` | por qué paradas pasa la L5, en orden |
| `tramos_ext.csv` | los tramos de la L5, con sus minutos |

⚠️ **El orden importa:** primero las paradas; después líneas, `PASA_POR` y tramos. Si `pasa_por` o `tramos` corren antes, el `MATCH` de P13 no encuentra nada y faltan relaciones **sin ningún error**.

Así se ven los archivos:

In [ ]:
for nombre in ["paradas_ext", "lineas_ext", "pasa_por_ext", "tramos_ext"]:
    print(f"── import/{nombre}.csv")
    print(Path(f"import/{nombre}.csv").read_text(encoding="utf-8"))

### Lo de la diapositiva — las paradas

El archivo trae P13, que es nueva, y P07 y P10, que **ya están**. Después de cargarlo, ¿13 o 15 paradas?

In [ ]:
CARGA_PARADAS = """
LOAD CSV WITH HEADERS FROM 'file:///paradas_ext.csv' AS row
MATCH (c:Comuna {nombre: row.comuna})
MERGE (p:Parada {id: row.id})
  ON CREATE SET p.nombre = row.nombre,
                p.lat = toFloat(row.lat),
                p.lon = toFloat(row.lon)
MERGE (p)-[:EN]->(c)
"""
q(CARGA_PARADAS)
q("MATCH (n:Parada) RETURN count(n) AS paradas")

**Esperado:** `paradas = 13`. Ahora la **misma** carga, otra vez:

In [ ]:
q(CARGA_PARADAS)
q("MATCH (n:Parada) RETURN count(n) AS paradas")

**Esperado:** 13 otra vez. Eso es idempotencia.

### Lo de la diapositiva — la línea nueva

In [ ]:
CARGA_LINEAS = """
LOAD CSV WITH HEADERS FROM 'file:///lineas_ext.csv' AS row
MERGE (l:Linea {id: row.id})
  ON CREATE SET l.nombre = row.nombre
"""
q(CARGA_LINEAS)
q("MATCH (l:Linea) RETURN count(l) AS lineas")

**Esperado:** `lineas = 5`.

### Ahora ustedes — por qué paradas pasa

In [ ]:
CARGA_PASA_POR = """
LOAD CSV WITH HEADERS FROM 'file:///pasa_por_ext.csv' AS row
MATCH (l:Linea {id: row.linea}), (p:Parada {id: row.parada})
___ (l)-[r:PASA_POR]->(p)
  ON CREATE SET r.orden = toInteger(row.orden)
"""   # Pista: el verbo que crea solo si no existe (con CREATE, recargar duplicaría)
q(CARGA_PASA_POR)
q("MATCH ()-[r:PASA_POR]->() RETURN count(r) AS pasa_por")

**Esperado:** `pasa_por = 19` (16 + 3).

### Lo de la diapositiva — los tramos

In [ ]:
CARGA_TRAMOS = """
LOAD CSV WITH HEADERS FROM 'file:///tramos_ext.csv' AS row
MATCH (a:Parada {id: row.desde}), (b:Parada {id: row.hasta})
MERGE (a)-[r:SIGUE_A {linea: row.linea}]->(b)
  ON CREATE SET r.minutos = toInteger(row.minutos)
"""
q(CARGA_TRAMOS)
q("MATCH ()-[r:SIGUE_A]->() RETURN count(r) AS sigue_a")

**Esperado:** `sigue_a = 14` (12 + 2).

### Lo de la diapositiva — verificar la carga

In [ ]:
display(q("MATCH ()-[r]->() RETURN type(r) AS relacion, count(*) AS n ORDER BY relacion"))
q("MATCH (n) RETURN labels(n)[0] AS tipo, count(*) AS n ORDER BY tipo")

**Esperado:** EN 13 · PASA_POR 19 · SIGUE_A 14, y Comuna 7 · Linea 5 · Parada 13. Si algo no calza, lo más probable es un CSV con otra codificación o una carga ejecutada antes que la de las paradas.

**Idempotencia:** las cuatro cargas otra vez, seguidas, y los mismos dos conteos.

In [ ]:
for carga in [CARGA_PARADAS, CARGA_LINEAS, CARGA_PASA_POR, CARGA_TRAMOS]:
    q(carga)
display(q("MATCH ()-[r]->() RETURN type(r) AS relacion, count(*) AS n ORDER BY relacion"))
q("MATCH (n) RETURN labels(n)[0] AS tipo, count(*) AS n ORDER BY tipo")

**Esperado:** los mismos números.

**✍️ Respuesta del equipo — 6:** Si en `pasa_por_ext` hubieran usado `CREATE` y recargado una vez, ¿cuántos `PASA_POR` habría? ¿Por qué la restricción de unicidad no lo impide? `___`

**✍️ Respuesta del equipo — 7:** ¿Por qué `ON CREATE SET` y no un `SET` a secas? `___`

---
# Ejercicio 6 — ¿Cambió la mejor ruta?

Las mismas consultas del Ej. 1 y del Ej. 2, de **Terminal Talcahuano** a **Plaza Penco**, ahora con la L5 cargada.

### Lo de la diapositiva

Por saltos:

In [ ]:
df = q("""
MATCH (a:Parada {id: 'P07'}), (b:Parada {id: 'P10'})
MATCH p = shortestPath((a)-[:SIGUE_A*..10]-(b))
RETURN [n IN nodes(p) | n.nombre] AS ruta, length(p) AS saltos
""")
nombres(df)
dibujar(df["ruta"][0])

Por minutos:

In [ ]:
df = q("""
MATCH p = (a:Parada {id: 'P07'})-[:SIGUE_A*..6]-(b:Parada {id: 'P10'})
WITH p, reduce(t = 0, r IN relationships(p) | t + r.minutos) AS minutos
RETURN [n IN nodes(p) | n.nombre] AS ruta, length(p) AS saltos, minutos
ORDER BY minutos
""")
nombres(df)

**Esperado:** por saltos, Terminal Talcahuano → Hospital Regional → Plaza Penco, **2 saltos** (en el dibujo aparece Hospital Regional). Por minutos, 2 filas: primero esa misma, con **50 min** (28 + 22), y después la de 5 saltos, con **55 min** (10 + 8 + 12 + 7 + 18).

### Ahora ustedes — ¿y sin usar la exprés?

`WHERE all(r IN relationships(p) WHERE …)` filtra los caminos **antes** de elegir el más corto.

In [ ]:
df = q("""
MATCH (a:Parada {id: 'P07'}), (b:Parada {id: 'P10'})
MATCH p = shortestPath((a)-[:SIGUE_A*..10]-(b))
WHERE all(r IN relationships(p) WHERE ___)
RETURN [n IN nodes(p) | n.nombre] AS ruta, length(p) AS saltos
""")   # Pista: ningún tramo del camino puede ser de la línea 'L5'
nombres(df)

**Esperado:** vuelve la ruta de 5 saltos del Ej. 1: Terminal Talcahuano → Parque Pedro del Río → Mall del Trébol → Plaza Perú → Barrio Norte → Plaza Penco.

**✍️ Respuesta del equipo — 8:** Ninguna consulta del Ej. 1 cambió y la respuesta sí. ¿Qué dice eso de guardar la red como grafo en vez de guardar las rutas ya calculadas? `___`

---
# Checkpoint

Vuelve a ejecutar las consultas clave y las compara con lo esperado. Es la tabla de la diapositiva 21. Corre **al final**, con la ampliación cargada, así que cada chequeo usa el valor del estado final.

In [ ]:
def ruta(a, b, extra=""):
    return int(q(f"""
    MATCH (x:Parada {{id: '{a}'}}), (y:Parada {{id: '{b}'}})
    MATCH p = shortestPath((x)-[:SIGUE_A*..10]-(y)) {extra}
    RETURN length(p) AS saltos""")["saltos"][0])

def min_minutos(a, b):
    return int(q(f"""
    MATCH p = (:Parada {{id: '{a}'}})-[:SIGUE_A*..6]-(:Parada {{id: '{b}'}})
    RETURN min(reduce(t = 0, r IN relationships(p) | t + r.minutos)) AS m""")["m"][0])

def ops(cypher):
    return [o.strip() for o in operadores(cypher)["operador"]]

chequeos = {
    "Etiquetas finales (Comuna, Linea, Parada)": (q("MATCH (n) RETURN labels(n)[0] AS t, count(*) AS n ORDER BY t")["n"].tolist(), [7, 5, 13]),
    "Relaciones finales (EN, PASA_POR, SIGUE_A)": (q("MATCH ()-[r]->() RETURN type(r) AS t, count(*) AS n ORDER BY t")["n"].tolist(), [13, 19, 14]),
    "Ej. 2 · U. de Concepción → Lomas Coloradas: saltos / mínimo de minutos": ((ruta("P02", "P12"), min_minutos("P02", "P12")), (2, 26)),
    "Ej. 3 · minutos de la L4": (int(q("MATCH ()-[r:SIGUE_A {linea:'L4'}]->() RETURN sum(r.minutos) AS m")["m"][0]), 40),
    "Ej. 4 · no alcanzables desde Coronel en ≤3 (con P13 ya cargada: 7; en el Ej. 4 eran 6)": (len(q("MATCH (o:Parada {id:'P11'}), (x:Parada) WHERE x <> o AND NOT EXISTS { MATCH (o)-[:SIGUE_A*..3]-(x) } RETURN x")), 7),
    "Ej. 5 · índice parada_nombre en uso": ("NodeIndexSeek" in ops("MATCH (p:Parada {nombre: 'Plaza Perú'}) RETURN p"), True),
    "Ej. 6 · Terminal Talcahuano → Plaza Penco: saltos / mínimo de minutos": ((ruta("P07", "P10"), min_minutos("P07", "P10")), (2, 50)),
    "Ej. 6 · sin la exprés: saltos": (ruta("P07", "P10", "WHERE all(r IN relationships(p) WHERE r.linea <> 'L5')"), 5),
}
for nombre, (obtenido, esperado) in chequeos.items():
    print("✓" if obtenido == esperado else "✗", nombre, "→", obtenido, "" if obtenido == esperado else f"(esperado {esperado})")

**Esperado:** 8 ✓. Si fallan las dos primeras líneas, repitan la Parte 2 en orden (paradas primero). Si falla «Ej. 5», el índice no se creó o no está `ONLINE`: repitan el paso 2 del Ej. 5.

**No bajen el contenedor todavía** si quieren mirar algo más en el Browser.

---
# Síntesis del equipo

Tres líneas:

1. En su **proyecto semestral**, una pregunta de «mejor camino»: ¿qué significaría «mejor» (saltos, minutos, costo…)? `___`
2. Una pregunta de su proyecto del tipo «¿qué **no** está conectado / cubierto?». `___`
3. Si tuvieran que cargar sus datos a Neo4j: ¿qué restricciones crearían **antes** de cargar y por qué? `___`

---
### Limpieza (al terminar)

`down -v` apaga el contenedor y borra el volumen (con el grafo, el índice y la ampliación). No se pierde nada que importe: el notebook reconstruye todo en ~1 minuto. Si prefieren dejarlo, `docker compose down` (sin `-v`) conserva el grafo.

In [ ]:
driver.close()
subprocess.run(["docker", "compose", "down", "-v"], check=True)   # sin -v, el grafo queda en el volumen
print("Listo.")

---
*Fin del Laboratorio 09.*